# Pipeline Open-Meteo: Observaciones + Predicciones Horarias

Descarga horaria desde Open-Meteo para todas las estaciones de `local_cleanup/clima_unificado.parquet`.

- **Observaciones**: `archive-api.open-meteo.com/v1/archive` (ERA5)
- **Predicciones**: `historical-forecast-api.open-meteo.com/v1/forecast` con modelo `gfs_seamless`
- **Rango**: 2025-10-01 → 2026-04-15 (horario, UTC)
- **Output**: `datos_historicos/historicos.parquet` (24 columnas, schema preservado)

> `visibility_observation` quedará 100% NaN: el archive API (ERA5) de Open-Meteo no expone esa variable.


## 1. Setup


In [ ]:
import os
import time
import logging
import threading
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests_cache
from retry_requests import retry
import openmeteo_requests
from tqdm.auto import tqdm

# --- Rango y rutas ---
START_DATE     = "2025-10-01"
END_DATE       = "2026-04-15"

REPO_ROOT      = Path.cwd().parent if Path.cwd().name == "datos_historicos" else Path.cwd()
INPUT_PARQUET  = REPO_ROOT / "local_cleanup" / "clima_unificado.parquet"
OUTPUT_DIR     = REPO_ROOT / "datos_historicos"
PARTIAL_DIR    = OUTPUT_DIR / "historicos_partiales"
CACHE_PATH     = OUTPUT_DIR / ".cache.sqlite"
OUTPUT_PARQUET = OUTPUT_DIR / "historicos.parquet"
COMPLETED_CSV  = PARTIAL_DIR / "estaciones_completadas.csv"
LOG_PATH       = OUTPUT_DIR / "pipeline.log"
QA_REPORT_PATH = OUTPUT_DIR / "reporte_calidad.txt"

# --- Parámetros operacionales ---
CHUNK_SIZE       = 25     # estaciones por chunk parquet
MAX_WORKERS      = 2      # reducido para no exceder límite por minuto
REQUEST_DELAY    = 0.5    # 2 rps = 120 req/min, bien bajo el límite de 600/min
RATE_LIMIT_SLEEP = 60     # pausa al recibir 429

# --- Endpoints Open-Meteo ---
ARCHIVE_URL    = "https://archive-api.open-meteo.com/v1/archive"
FORECAST_URL   = "https://historical-forecast-api.open-meteo.com/v1/forecast"
FORECAST_MODEL = "gfs_seamless"

# Orden importa: openmeteo_requests devuelve Variables(i) por posición.
OPENMETEO_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "pressure_msl",
    "surface_pressure",
    "visibility",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "is_day",
]

# Open-Meteo → nombre base en schema final
VAR_RENAME = {
    "temperature_2m":       "temp",
    "relative_humidity_2m": "hum",
    "dew_point_2m":         "dew",
    "precipitation":        "precip_1h",
    "pressure_msl":         "sea_level_press",
    "surface_pressure":     "press",
    "visibility":           "visibility",
    "wind_speed_10m":       "wind_speed",
    "wind_direction_10m":   "wind_dir",
    "wind_gusts_10m":       "wind_gust",
    "is_day":               "is_daytime",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler(LOG_PATH, mode="a", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger("openmeteo-pipeline")
log.info("Setup completo. START=%s END=%s OUT=%s", START_DATE, END_DATE, OUTPUT_PARQUET)


## 2. Reset selectivo

Backup defensivo de `historicos.parquet` previo + limpieza de cache y chunks viejos. Idempotente: se puede correr múltiples veces.


In [ ]:
def reset_pipeline_state():
    ts = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    if OUTPUT_PARQUET.exists():
        bak = OUTPUT_PARQUET.with_name(f"historicos.parquet.bak.{ts}")
        OUTPUT_PARQUET.rename(bak)
        log.info("Backup parquet previo → %s", bak.name)
    if CACHE_PATH.exists():
        try:
            CACHE_PATH.unlink()
            log.info("Cache SQLite eliminado.")
        except PermissionError:
            # En Windows el archivo puede estar bloqueado por una sesión anterior.
            # Renombramos en lugar de borrar; se creará uno nuevo al iniciar la sesión.
            bak_cache = CACHE_PATH.with_name(f".cache.bak.{ts}.sqlite")
            try:
                CACHE_PATH.rename(bak_cache)
                log.warning("Cache SQLite bloqueado; renombrado → %s. Si persiste el error, reiniciar kernel.", bak_cache.name)
            except Exception as e2:
                log.warning("No se pudo eliminar ni renombrar el cache SQLite: %s. Continuando sin resetear cache.", e2)
    if PARTIAL_DIR.exists():
        for p in PARTIAL_DIR.glob("chunk_*.parquet"):
            p.unlink()
        if COMPLETED_CSV.exists():
            COMPLETED_CSV.unlink()
        log.info("Chunks parciales y estaciones_completadas.csv borrados.")
    PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

# Ejecutar reset al inicio del run.
reset_pipeline_state()


## 3. Carga y validación de estaciones

Lee `clima_unificado.parquet`, deduplica por `station_id` y descarta coordenadas inválidas.


In [10]:
df_estaciones = pd.read_parquet(INPUT_PARQUET)

cols_estacion = [c for c in ["station_id", "lat_estacion", "lon_estacion", "estado", "zona_id"] if c in df_estaciones.columns]
df_estaciones = (
    df_estaciones[cols_estacion]
    .drop_duplicates(subset=["station_id"])
    .reset_index(drop=True)
)

mask_validas = (
    df_estaciones["lat_estacion"].between(-90, 90)
    & df_estaciones["lon_estacion"].between(-180, 180)
    & df_estaciones["lat_estacion"].notna()
    & df_estaciones["lon_estacion"].notna()
)
n_invalidas = int((~mask_validas).sum())
if n_invalidas:
    log.warning("Descartando %d estaciones con coordenadas inválidas.", n_invalidas)
df_estaciones = df_estaciones[mask_validas].reset_index(drop=True)

log.info("Estaciones a procesar: %d", len(df_estaciones))
df_estaciones.head()


2026-05-18 22:34:09,183 INFO Estaciones a procesar: 522


,station_id,lat_estacion,lon_estacion,estado,zona_id
0,KXBP,33.17528,-97.82833,TX,ZONA_02
1,KPVW,34.16806,-101.71722,TX,ZONA_20
2,KCQB,35.72389,-96.82028,OK,ZONA_06
3,KVYS,41.35175,-89.14963,IL,ZONA_03
4,KOJA,35.54472,-98.66833,OK,ZONA_06


## 4. Sesión Open-Meteo robusta

Cache SQLite persistente + retry exponencial + throttling global por lock.


In [11]:
# requests_cache añade el sufijo .sqlite automáticamente; usamos el stem.
cached_session = requests_cache.CachedSession(
    str(CACHE_PATH.with_suffix("")),
    expire_after=-1,
)
retry_session = retry(cached_session, retries=5, backoff_factor=2)
om_client = openmeteo_requests.Client(session=retry_session)

# Throttle global: garantiza ≥ REQUEST_DELAY entre cualquier par de requests.
_throttle_lock = threading.Lock()
_last_request_ts = [0.0]

def _throttle():
    with _throttle_lock:
        elapsed = time.monotonic() - _last_request_ts[0]
        wait = REQUEST_DELAY - elapsed
        if wait > 0:
            time.sleep(wait)
        _last_request_ts[0] = time.monotonic()

log.info("Sesión Open-Meteo lista (cache=%s).", CACHE_PATH.name)


2026-05-18 22:34:12,705 INFO Sesión Open-Meteo lista (cache=.cache.sqlite).


## 5. Funciones de descarga

Convierte la respuesta binaria de `openmeteo_requests` a un DataFrame horario tidy.


In [12]:
def _responses_to_df(responses, suffix: str, station_id: str) -> pd.DataFrame:
    if not responses:
        return pd.DataFrame()
    response = responses[0]
    hourly = response.Hourly()
    if hourly is None:
        return pd.DataFrame()

    start = pd.to_datetime(hourly.Time(), unit="s", utc=True)
    end   = pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True)
    step  = pd.Timedelta(seconds=hourly.Interval())
    timestamps = pd.date_range(start=start, end=end, freq=step, inclusive="left")

    data = {"time": timestamps, "station_id": station_id}
    for i, om_name in enumerate(OPENMETEO_VARS):
        values = hourly.Variables(i).ValuesAsNumpy()
        base = VAR_RENAME[om_name]
        data[f"{base}_{suffix}"] = values
    return pd.DataFrame(data)


def _params(lat: float, lon: float) -> dict:
    return {
        "latitude": float(lat),
        "longitude": float(lon),
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": OPENMETEO_VARS,
        "timezone": "UTC",
        "wind_speed_unit": "kmh",
    }


def descargar_observaciones(station_id: str, lat: float, lon: float) -> pd.DataFrame:
    _throttle()
    responses = om_client.weather_api(ARCHIVE_URL, params=_params(lat, lon))
    return _responses_to_df(responses, suffix="observation", station_id=station_id)


def descargar_predicciones(station_id: str, lat: float, lon: float) -> pd.DataFrame:
    _throttle()
    params = _params(lat, lon)
    params["models"] = FORECAST_MODEL
    responses = om_client.weather_api(FORECAST_URL, params=params)
    return _responses_to_df(responses, suffix="prediction", station_id=station_id)


## 6. Merge por estación

Inner join `obs` × `pred` por `(station_id, time)`. Castea a `float32` para coincidir con el schema previo.


In [13]:
SCHEMA_FLOAT_COLS = [
    "temp_observation", "dew_observation", "hum_observation", "precip_1h_observation",
    "sea_level_press_observation", "press_observation", "wind_speed_observation",
    "wind_dir_observation", "wind_gust_observation", "visibility_observation",
    "is_daytime_observation",
    "temp_prediction", "dew_prediction", "hum_prediction", "precip_1h_prediction",
    "sea_level_press_prediction", "press_prediction", "wind_speed_prediction",
    "wind_dir_prediction", "wind_gust_prediction", "visibility_prediction",
    "is_daytime_prediction",
]

ORDER_COLS = ["time", "station_id"] + SCHEMA_FLOAT_COLS


def merge_estacion(obs_df: pd.DataFrame, pred_df: pd.DataFrame) -> pd.DataFrame:
    if obs_df.empty or pred_df.empty:
        return pd.DataFrame(columns=ORDER_COLS)
    merged = obs_df.merge(pred_df, on=["time", "station_id"], how="inner")
    for col in SCHEMA_FLOAT_COLS:
        if col not in merged.columns:
            merged[col] = np.nan
    merged = merged[ORDER_COLS].copy()
    merged[SCHEMA_FLOAT_COLS] = merged[SCHEMA_FLOAT_COLS].astype("float32")
    # Resolución ms con tz UTC, idéntica al parquet previo.
    merged["time"] = pd.to_datetime(merged["time"], utc=True).astype("datetime64[ms, UTC]")
    return merged


## 7. Smoke test (1 estación)

Antes del full run, prueba con la primera estación para validar schema, endpoints y unidades.


In [14]:
smoke = df_estaciones.iloc[0]
log.info("Smoke test: station_id=%s lat=%s lon=%s", smoke["station_id"], smoke["lat_estacion"], smoke["lon_estacion"])

obs_smoke  = descargar_observaciones(smoke["station_id"], smoke["lat_estacion"], smoke["lon_estacion"])
pred_smoke = descargar_predicciones(smoke["station_id"], smoke["lat_estacion"], smoke["lon_estacion"])
merged_smoke = merge_estacion(obs_smoke, pred_smoke)

print("obs:", obs_smoke.shape, "pred:", pred_smoke.shape, "merged:", merged_smoke.shape)
print("\ndtypes:")
print(merged_smoke.dtypes)
merged_smoke.head(3)


2026-05-18 22:34:19,840 INFO Smoke test: station_id=KXBP lat=33.17528 lon=-97.82833


OpenMeteoRequestsError: failed to request 'https://historical-forecast-api.open-meteo.com/v1/forecast': {'reason': "Data corrupted at path ''. Cannot initialize MultiDomains from invalid String value ncep_gfs_seamless.", 'error': True}

## 8. Descarga masiva con `ThreadPoolExecutor` + checkpoints

Resume automático: lee `estaciones_completadas.csv` y procesa solo las pendientes. Persiste un parquet cada `CHUNK_SIZE` estaciones.


In [ ]:
def cargar_completadas() -> set:
    if COMPLETED_CSV.exists():
        return set(pd.read_csv(COMPLETED_CSV)["station_id"].astype(str))
    return set()


def appendear_completadas(station_ids: list):
    if not station_ids:
        return
    is_new = not COMPLETED_CSV.exists()
    pd.DataFrame({"station_id": station_ids}).to_csv(
        COMPLETED_CSV, mode="a", header=is_new, index=False
    )


def proximo_chunk_index() -> int:
    existentes = sorted(PARTIAL_DIR.glob("chunk_*.parquet"))
    if not existentes:
        return 1
    nums = [int(p.stem.split("_")[1]) for p in existentes]
    return max(nums) + 1


def persistir_chunk(rows, station_ids):
    if not rows:
        return
    chunk_df = pd.concat(rows, ignore_index=True)
    idx = proximo_chunk_index()
    out_path = PARTIAL_DIR / f"chunk_{idx:03d}.parquet"
    chunk_df.to_parquet(out_path, compression="snappy", index=False)
    appendear_completadas(station_ids)
    log.info("Chunk persistido: %s (%d filas, %d estaciones).", out_path.name, len(chunk_df), len(station_ids))


def procesar_estacion(row):
    sid = row["station_id"]
    t0 = time.monotonic()
    try:
        obs = descargar_observaciones(sid, row["lat_estacion"], row["lon_estacion"])
        pred = descargar_predicciones(sid, row["lat_estacion"], row["lon_estacion"])
        merged = merge_estacion(obs, pred)
        log.info("OK %s rows=%d latencia=%.2fs", sid, len(merged), time.monotonic() - t0)
        return sid, merged, None
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        log.error("FAIL %s %s", sid, msg)
        return sid, None, msg


completadas = cargar_completadas()
pendientes = df_estaciones[~df_estaciones["station_id"].astype(str).isin(completadas)].reset_index(drop=True)
log.info("Pendientes: %d / %d", len(pendientes), len(df_estaciones))

buffer_rows = []
buffer_sids = []
fallidas = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(procesar_estacion, row): row["station_id"] for _, row in pendientes.iterrows()}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Estaciones"):
        sid, df, err = fut.result()
        if err is not None or df is None or df.empty:
            fallidas.append((sid, err or "df vacío"))
            continue
        buffer_rows.append(df)
        buffer_sids.append(sid)
        if len(buffer_sids) >= CHUNK_SIZE:
            persistir_chunk(buffer_rows, buffer_sids)
            buffer_rows, buffer_sids = [], []

persistir_chunk(buffer_rows, buffer_sids)
log.info("Descarga terminada. Fallidas=%d", len(fallidas))
if fallidas:
    print("\nFallidas (primeras 20):")
    for s, e in fallidas[:20]:
        print(f"  {s}: {e}")


## 9. Consolidación a `historicos.parquet`

Concatena todos los chunks, deduplica y ordena. Escribe el parquet final con compresión snappy.


In [ ]:
chunks = sorted(PARTIAL_DIR.glob("chunk_*.parquet"))
if not chunks:
    raise RuntimeError("No hay chunks para consolidar.")

dfs = [pd.read_parquet(p) for p in chunks]
df_final = pd.concat(dfs, ignore_index=True)
df_final = df_final.drop_duplicates(subset=["station_id", "time"], keep="last")
df_final = df_final.sort_values(["station_id", "time"]).reset_index(drop=True)
df_final.to_parquet(OUTPUT_PARQUET, compression="snappy", index=False)
log.info("Consolidado %d filas → %s", len(df_final), OUTPUT_PARQUET.name)
print("shape:", df_final.shape)
df_final.head(3)


## 10. Validaciones de calidad

Cobertura por estación, duplicados, NaNs, rangos físicos. Reporte a stdout y a `reporte_calidad.txt`.


In [ ]:
HORAS_ESPERADAS = 4728  # 197 días × 24h

def reporte_calidad(df: pd.DataFrame) -> str:
    lineas = []
    lineas.append(f"Filas totales: {len(df):,}")
    lineas.append(f"Estaciones únicas: {df['station_id'].nunique()}")
    lineas.append(f"Rango temporal: {df['time'].min()} → {df['time'].max()}")

    cobertura = df.groupby("station_id").size()
    lineas.append(f"Cobertura horaria — min={cobertura.min()} mediana={cobertura.median():.0f} max={cobertura.max()} esperada={HORAS_ESPERADAS}")
    incompletas = cobertura[cobertura < int(0.95 * HORAS_ESPERADAS)]
    lineas.append(f"Estaciones con <95% cobertura: {len(incompletas)}")
    if len(incompletas):
        muestra = ", ".join(incompletas.index.astype(str).tolist()[:20])
        lineas.append(f"  ej: {muestra}{'...' if len(incompletas) > 20 else ''}")

    dups = int(df.duplicated(subset=["station_id", "time"]).sum())
    lineas.append(f"Duplicados (station_id, time): {dups}")

    lineas.append("\nPorcentaje de NaN por columna:")
    nan_pct = (df.isna().mean().sort_values(ascending=False) * 100)
    for col, pct in nan_pct.items():
        if pct > 0:
            lineas.append(f"  {col}: {pct:.1f}%")

    lineas.append("\nRangos físicos:")
    checks = [
        ("temp_observation", -60, 60),
        ("temp_prediction", -60, 60),
        ("hum_observation", 0, 100),
        ("hum_prediction", 0, 100),
        ("wind_speed_observation", 0, 500),
        ("wind_speed_prediction", 0, 500),
        ("precip_1h_observation", 0, 500),
        ("precip_1h_prediction", 0, 500),
    ]
    for col, lo, hi in checks:
        if col in df.columns:
            s = df[col].dropna()
            if len(s) == 0:
                lineas.append(f"  {col}: vacío")
                continue
            fuera = int(((s < lo) | (s > hi)).sum())
            lineas.append(f"  {col}: min={s.min():.2f} max={s.max():.2f} fuera_de_rango={fuera}")
    return "\n".join(lineas)


reporte = reporte_calidad(df_final)
print(reporte)
QA_REPORT_PATH.write_text(reporte, encoding="utf-8")
log.info("Reporte de calidad guardado en %s", QA_REPORT_PATH.name)
